# Laptop2 StackOverflow run — clean and export

Cleans the full SQLStorm StackOverflow run on laptop2 (`so_uniform_1_16_20260930T094325Z`: every query
once, batch sizes {1, 16}, 2 warm-ups, 15 s statement timeout, 2.5 GHz pin, 40–60 °C thermal gate) and
writes a folder that can be handed to someone else on its own:

| file | content |
|---|---|
| `query_energy.csv` | every fully successful query (no timeout in any batch) and its energy per execution from the slope |
| `excluded_queries.csv` | the queries left out and why |
| `query_execution_stats.csv` | PostgreSQL execution statistics of each successful query |
| `duplicates_and_similar.csv` | exact duplicate queries, near-duplicates and queries with identical output |
| `duplicate_and_similar_groups.csv` | the same-output relation as one row per group, with the SQL variants inside it |
| `empty_output_queries.csv` | queries whose result is empty, with the likely reason |
| `run_info.csv` | run parameters, machine, OS, PostgreSQL and dataset provenance |
| `database_tables.csv` | the database's tables, row counts and sizes |
| `README.md` | method, column dictionary and caveats |

Inputs: the run folder on laptop2 (pulled to `cache/laptop2_cache/`), the query files
(`cache/sqlstorm_cache/stackoverflow/queries/`, from `sqlstorm_analysis.ipynb`), the query outputs
(`cache/outputs_cache/`, from `sqlstorm_outputs.ipynb`) and the query feature table
(`csv/sqlstorm_stackoverflow_query_features.csv`).

In [1]:
# %pip install paramiko pandas numpy
import io, re, json, zlib, hashlib, tarfile, time, itertools, collections, difflib
from pathlib import Path
import numpy as np
import pandas as pd

SSH_HOST = "192.168.1.59"                       # laptop2
SSH_USER = "user01"
SSH_PASS = "a"                                   # demo credential
REMOTE_ROOT = "/home/user01/GreenSQL"
RUN_ID = "so_uniform_1_16_20260930T094325Z"
DB = "stackoverflow_1gb"
CACHE = Path("cache/laptop2_cache")
RUN = CACHE / RUN_ID
QDIR = Path("cache/sqlstorm_cache/stackoverflow/queries")
ODIR = Path("cache/outputs_cache/stackoverflow_1gb")
OUT_CONSOLE = Path("cache/outputs_cache/stackoverflow_1gb.console.log")
FEATURES = Path("csv/sqlstorm_stackoverflow_query_features.csv")
EXPORT = Path("export/sqlstorm_stackoverflow_energy_laptop2")
SYSINFO_CMD = ("hostname; echo '##'; cat /etc/os-release; echo '##'; uname -r; echo '##'; free -b; echo '##'; "
               "lsblk -d -o NAME,MODEL,SIZE,ROTA; echo '##'; grep -vE '^\\s*(#|$)' /etc/postgresql/18/main/postgresql.conf; "
               "echo '##'; ls -A /etc/postgresql/18/main/conf.d; echo '##'; "
               f"cd {REMOTE_ROOT}/queries/stackoverflow/SQLStorm && md5sum *.sql")


def pull():
    RUN.parent.mkdir(parents=True, exist_ok=True)
    try:
        import paramiko
        cl = paramiko.SSHClient()
        cl.set_missing_host_key_policy(paramiko.AutoAddPolicy())
        cl.connect(SSH_HOST, username=SSH_USER, password=SSH_PASS, timeout=45, banner_timeout=45, auth_timeout=45)
        try:
            _, out, _ = cl.exec_command(f"tar -C {REMOTE_ROOT}/logs/warm_stepup -czf - {RUN_ID}")
            with tarfile.open(fileobj=out, mode="r|gz") as tf:
                tf.extractall(CACHE, filter="data")
            _, out, _ = cl.exec_command(SYSINFO_CMD)
            (CACHE / f"{RUN_ID}.sysinfo.txt").write_bytes(out.read())
            return "server"
        finally:
            cl.close()
    except Exception as exc:
        if not (RUN / "summary.txt").exists():
            raise RuntimeError(f"{exc}\nNo cache at '{RUN}/'.") from exc
        return f"cache ({(time.time() - (RUN / 'summary.txt').stat().st_mtime) / 3600:.0f} h old; pull failed: {type(exc).__name__})"


print("laptop2 run:", pull())
for p, what in [(QDIR, "query files"), (ODIR, "query outputs"), (FEATURES, "feature table")]:
    assert p.exists(), f"missing {what} at {p} — run the notebook that produces it first"
print(sorted(f.name for f in RUN.iterdir()))

laptop2 run: server
['console.log', 'query_catalog_stackoverflow_1gb.csv', 'query_runinfo_stackoverflow_1gb.csv', 'query_samples_stackoverflow_1gb.csv', 'query_timing_stackoverflow_1gb.csv', 'run_order_so_uniform_1_16_20260930T094325Z.txt', 'summary.txt']


## 1. Clean: fully successful queries and the energy slope

Each query was run as 2 single-copy warm-ups, then a measured batch of 1 copy and a measured batch of
16 back-to-back copies. A query is **fully successful** when none of these four batches failed. With
`WARMUP_FAIL_SKIP` a warm-up timeout skips the measured batches.

Energy per execution is the **slope** `(E16 − E1) / 15`, from the RAPL package energy of the two
batches. It removes everything paid once per batch: starting the client, connecting, and the batch's
idle edges. Time per execution is the same slope on wall time.

In [2]:
T = pd.read_csv(RUN / f"query_timing_{DB}.csv")
T["query"] = T["query"].str.extract(r"(\d+)\.sql$")[0].astype(int)
ORDER = [int(re.search(r"(\d+)\.sql", l).group(1)) for l in (RUN / f"run_order_{RUN_ID}.txt").read_text().splitlines()
         if l.strip() and not l.startswith("#")]
ALLQ = sorted(set(ORDER))
SUMMARY = (RUN / "summary.txt").read_text()

b = T.groupby("query").agg(n_batches=("failed", "size"), n_failed=("failed", "sum"))
wf = T[(T["phase"] == "warmup") & (T["failed"] == 1)].groupby("query")["batch_index"].min()
mf = T[(T["phase"] == "measured") & (T["failed"] == 1)].groupby("query")["batchnum"].min()
b["reason"] = ""
b.loc[wf.index, "reason"] = [f"warm-up {i} timed out at 15 s (measured batches skipped)" for i in wf]
b.loc[mf.index, "reason"] = [f"measured N={n} batch failed (timeout)" for n in mf]
OK = b.index[b["n_failed"] == 0]
assert set(b.index) == set(ALLQ), "timing log does not cover the run order"
print(f"{len(ALLQ)} queries in the run;  fully successful: {len(OK)};  excluded: {len(ALLQ) - len(OK)}")
print(b.loc[b["n_failed"] > 0, "reason"].str.replace(r"warm-up \d", "warm-up", regex=True).value_counts().to_string())

m = T[(T["phase"] == "measured") & T["query"].isin(OK)]
P = m.pivot_table(index="query", columns="batchnum",
                  values=["rapl_pkg_j", "rapl_core_j", "rapl_dram_j", "elapsed_sec", "pkg_temp_start_c", "mhz_mean"])
assert P[("rapl_pkg_j", 1)].notna().all() and P[("rapl_pkg_j", 16)].notna().all()
Q = pd.DataFrame(index=P.index)
Q["energy_j"] = (P["rapl_pkg_j"][16] - P["rapl_pkg_j"][1]) / 15
Q["time_s"] = (P["elapsed_sec"][16] - P["elapsed_sec"][1]) / 15
Q["energy_core_j"] = (P["rapl_core_j"][16] - P["rapl_core_j"][1]) / 15
Q["energy_dram_j"] = (P["rapl_dram_j"][16] - P["rapl_dram_j"][1]) / 15
Q["fixed_overhead_j"] = P["rapl_pkg_j"][1] - Q["energy_j"]
Q["e1_j"], Q["e16_j"] = P["rapl_pkg_j"][1], P["rapl_pkg_j"][16]
Q["t1_s"], Q["t16_s"] = P["elapsed_sec"][1], P["elapsed_sec"][16]
Q["power_w"] = Q["e16_j"] / Q["t16_s"]
Q["temp_start_c"] = P["pkg_temp_start_c"][16]
Q["mhz_mean"] = P["mhz_mean"][16]
print(f"energy per execution: median {Q['energy_j'].median():.2f} J, range {Q['energy_j'].min():.3f}–"
      f"{Q['energy_j'].max():.1f} J;  slopes <= 0: energy {int((Q['energy_j'] <= 0).sum())}, time "
      f"{int((Q['time_s'] <= 0).sum())};  clock {Q['mhz_mean'].median():.0f} MHz median")

11794 queries in the run;  fully successful: 11431;  excluded: 363
reason
warm-up timed out at 15 s (measured batches skipped)    361
measured N=16 batch failed (timeout)                      2
energy per execution: median 3.97 J, range 0.012–225.9 J;  slopes <= 0: energy 0, time 0;  clock 1444 MHz median


### 1a. How precise is one slope?

The query set contains exact copies of the same SQL under different numbers (§2), and each copy was
measured independently in this run. Their spread is the repeat noise of a single slope on this
machine. A model `σ(b) = √(a² + (k·b)²)` is fitted to it and gives every query an estimated standard
error.

In [3]:
def norm_sql(txt):
    s = re.sub(r"--[^\n]*", " ", txt)
    s = re.sub(r"/\*.*?\*/", " ", s, flags=re.S)
    s = re.sub(r"(?is)^\s*EXPLAIN\s*\([^)]*\)", " ", s)
    return re.sub(r"\s+", " ", s).strip().rstrip(";").strip().lower()


SQL_RAW = {q: (QDIR / f"{q}.sql").read_text("utf-8", "replace") for q in ALLQ}
SQLN = pd.Series({q: norm_sql(t) for q, t in SQL_RAW.items()})
SQLH = SQLN.map(lambda s: hashlib.md5(s.encode()).hexdigest())

d = pd.DataFrame({"b": Q["energy_j"], "h": SQLH.reindex(Q.index)})
d = d[d.groupby("h")["b"].transform("size") > 1]
n_ = d.groupby("h")["b"].transform("size")
mb = d.groupby("h")["b"].transform("mean")
r2 = ((d["b"] - mb) ** 2 * n_ / (n_ - 1)).to_numpy()            # unbiased squared deviations
X = np.c_[np.ones(len(mb)), mb.to_numpy() ** 2]
th = np.linalg.lstsq(X, r2, rcond=None)[0]
for _ in range(20):                                              # IRLS: weight by the fitted variance
    w = 1 / np.maximum(X @ th, 1e-12) ** 2
    th = np.linalg.lstsq(X * np.sqrt(w)[:, None], r2 * np.sqrt(w), rcond=None)[0]
NOISE_A, NOISE_K = np.sqrt(max(th[0], 0)), np.sqrt(max(th[1], 0))
Q["energy_j_se"] = np.sqrt(NOISE_A ** 2 + (NOISE_K * Q["energy_j"]) ** 2)
NOISE_N, NOISE_SETS, NOISE_MAX = len(d), d["h"].nunique(), mb.max()
z = (d["b"] - mb) * np.sqrt(n_ / (n_ - 1)) / np.sqrt(NOISE_A ** 2 + (NOISE_K * mb) ** 2)
NOISE_TAIL = (z.abs() > 3).mean()                               # share of copies beyond 3 model sigmas
NOISE_REL_MAX = (d.groupby("h")["b"].std() / d.groupby("h")["b"].mean()).max()
print(f"noise model from {NOISE_N} measured copies in {NOISE_SETS} sets: a = {1000 * NOISE_A:.1f} mJ, "
      f"k = {100 * NOISE_K:.2f}%  (fitted on slopes up to {NOISE_MAX:.1f} J; extrapolated above);  "
      f"{100 * NOISE_TAIL:.1f}% of copies lie beyond 3 sigma (largest set CV {100 * NOISE_REL_MAX:.1f}%)")
for lo, hi in [(0, 1), (1, 2), (2, 5), (5, 20)]:
    s = (mb >= lo) & (mb < hi)
    if s.sum():
        print(f"  {lo}-{hi} J: observed SD {np.sqrt(r2[s.to_numpy()].mean()) * 1000:.1f} mJ, model "
              f"{1000 * np.sqrt(NOISE_A ** 2 + (NOISE_K * mb[s]) ** 2).mean():.1f} mJ  (n={int(s.sum())})")

noise model from 2727 measured copies in 452 sets: a = 0.8 mJ, k = 0.89%  (fitted on slopes up to 12.0 J; extrapolated above);  0.6% of copies lie beyond 3 sigma (largest set CV 10.2%)
  0-1 J: observed SD 4.8 mJ, model 5.2 mJ  (n=1221)
  1-2 J: observed SD 13.0 mJ, model 12.5 mJ  (n=365)
  2-5 J: observed SD 25.3 mJ, model 28.9 mJ  (n=1068)
  5-20 J: observed SD 257.4 mJ, model 81.9 mJ  (n=73)


## 2. Duplicates and similar queries

Relations between queries, in form (the SQL text) and in output, over all 11,794 queries of the run.
They are reported, not acted on: nothing is removed or merged.

* **duplicate** — identical SQL once normalised (SQLStorm header comment and the `EXPLAIN (…)`
  wrapper removed, comments removed, whitespace collapsed, lower-cased). The lowest-numbered query of a
  set is its canonical query.
* **similar** — different SQL with token 3-gram Jaccard similarity ≥ 0.8 (MinHash LSH for
  candidates, exact Jaccard to confirm). Similar queries often differ in one clause (a LIMIT, a column,
  a filter).
* **same output** — byte-identical result files in the output run (header, all rows, row count).

In [4]:
J_MIN = 0.8
canon = SQLN.reset_index().groupby(0)["index"].min()            # text -> canonical (lowest) query
CANON = SQLN.map(canon)                                          # query -> canonical query
U = CANON.unique()
toks = {q: re.findall(r"\w+|[^\w\s]", SQLN[q]) for q in U}
SH = {q: {zlib.crc32(" ".join(t[i:i + 3]).encode()) for i in range(max(1, len(t) - 2))} for q, t in toks.items()}
rng = np.random.default_rng(20261004)
NP_, BANDS, ROWS = 128, 16, 8
A_ = rng.integers(1, 2 ** 63, NP_, dtype=np.uint64) | np.uint64(1)
B_ = rng.integers(0, 2 ** 63, NP_, dtype=np.uint64)
SIG = {q: ((np.fromiter(s, np.uint64)[:, None] * A_ + B_) >> np.uint64(16)).min(axis=0) for q, s in SH.items()}
cand = set()
for bnd in range(BANDS):
    buckets = collections.defaultdict(list)
    for q, v in SIG.items():
        buckets[v[bnd * ROWS:(bnd + 1) * ROWS].tobytes()].append(q)
    for qs in buckets.values():
        cand.update(itertools.combinations(sorted(qs), 2))
NEAR = pd.DataFrame([(a, c, len(SH[a] & SH[c]) / len(SH[a] | SH[c])) for a, c in cand], columns=["a", "b", "jaccard"])
NEAR = NEAR[NEAR["jaccard"] >= J_MIN].reset_index(drop=True)

# byte-identical outputs (output run)
def out_md5(q):
    p = ODIR / f"{q}.txt"
    raw = p.read_bytes()
    return None if raw.startswith(b"psql:") else hashlib.md5(raw).hexdigest()


OUTH = pd.Series({q: out_md5(q) for q in ALLQ})


best = pd.concat([NEAR[["a", "b", "jaccard"]], NEAR.rename(columns={"a": "b", "b": "a"})[["a", "b", "jaccard"]]])
best = best.sort_values("jaccard", ascending=False).drop_duplicates("a").set_index("a")
dup_n = CANON.map(CANON.value_counts())
og = OUTH.dropna()
og_n = og.map(og.value_counts())
og_id = og[og_n > 1]                                             # queries sharing their output with another


def others(groups):
    """query -> ';'-joined other members of its group (groups: Series query -> group key)"""
    mem = pd.Series(groups.index, index=groups.values).groupby(level=0).apply(sorted)
    return pd.Series({q: ";".join(str(x) for x in mem[k] if x != q) for q, k in groups.items()
                      if len(mem[k]) > 1}, dtype="object")

DS = pd.DataFrame(index=pd.Index(ALLQ, name="query"))
DS["in_query_energy"] = DS.index.isin(OK)
DS["sql_hash"] = SQLH
DUP_OTHERS = others(CANON)
DS["duplicate_queries"] = DUP_OTHERS
DS["duplicate_count"] = DUP_OTHERS.str.count(";") + 1
members = CANON.reset_index().groupby(0)["index"].apply(list)          # canonical -> all its queries
nbr = collections.defaultdict(set)
for a_, c_ in NEAR[["a", "b"]].itertuples(index=False):
    nbr[a_].update(members[c_]); nbr[c_].update(members[a_])
sim_list = CANON.map(lambda c: ";".join(map(str, sorted(nbr[c]))) if c in nbr else pd.NA)
DS["similar_queries"] = sim_list
DS["similar_count"] = CANON.map(lambda c: len(nbr[c]) if c in nbr else pd.NA)
DS["most_similar_query"] = CANON.map(best["b"])
DS["most_similar_jaccard"] = CANON.map(best["jaccard"]).round(3)
OUT_OTHERS = others(og_id)
DS["same_output_queries"] = OUT_OTHERS
DS["same_output_count"] = OUT_OTHERS.str.count(";") + 1
for c in ["duplicate_count", "similar_count", "most_similar_query", "same_output_count"]:
    DS[c] = pd.to_numeric(DS[c], errors="coerce").astype("Int64")
KEEP = DS["duplicate_count"].notna() | DS["similar_count"].notna() | DS["same_output_count"].notna()
DUPSIM = DS[KEEP]
print(f"distinct SQL texts: {len(U)} for {len(ALLQ)} queries;  duplicate sets: {int((CANON.value_counts() > 1).sum())} "
      f"({int((dup_n > 1).sum())} queries, {len(ALLQ) - len(U)} redundant copies)")
print(f"similar pairs (Jaccard >= {J_MIN}): {len(NEAR)} between {NEAR[['a', 'b']].stack().nunique()} distinct texts; "
      f"queries with a similar query: {int(DS['similar_count'].notna().sum())} (most neighbours {int(DS['similar_count'].max())})")
print(f"same-output groups: {og_id.nunique()} ({len(og_id)} queries);  rows in duplicates_and_similar.csv: {len(DUPSIM)}")

distinct SQL texts: 9519 for 11794 queries;  duplicate sets: 452 (2727 queries, 2275 redundant copies)
similar pairs (Jaccard >= 0.8): 12958 between 1570 distinct texts; queries with a similar query: 3834 (most neighbours 505)
same-output groups: 452 (3050 queries);  rows in duplicates_and_similar.csv: 3921


## 3. Empty-output queries

From the output run (`output_runner`: every query once, 60 s timeout, result rows saved). That run was
on laptop1, a second Latitude 7490 with the same database: §4 checks that every table has the same
size on both machines, and the query files are byte-identical (checked against laptop2's copies). A
result is empty when psql reported `(0 rows)`. The likely reason is the first matching feature of the
SQL; `wallclock_dependent` marks queries that filter on `NOW()` / `CURRENT_DATE`, whose result depends
on the day they are run (the data ends in 2024).

In [5]:
F = pd.read_csv(FEATURES).set_index("qid")
WALL = re.compile(r"\bNOW\s*\(|\bCURRENT_(DATE|TIMESTAMP)\b|\bLOCALTIMESTAMP\b", re.I)
nc = {q: re.sub(r"--[^\n]*", " ", SQL_RAW[q]) for q in ALLQ}


def min_window_days(s):
    unit = {"day": 1, "days": 1, "week": 7, "weeks": 7, "month": 30.4, "months": 30.4, "year": 365, "years": 365}
    v = [int(n) * unit[u.lower()] for n, u in re.findall(r"INTERVAL\s*'(\d+)\s*([A-Za-z]+)'", s) if u.lower() in unit]
    return min(v) if v else np.nan


def out_status(q):
    t = (ODIR / f"{q}.txt").read_text("utf-8", "replace")
    if t.startswith("psql:"):
        return "timeout", np.nan
    m = re.search(r"\n\((\d+) rows?\)\n?$", t)
    return ("complete", int(m.group(1))) if m else ("capped at 1000 rows", np.nan)


OS = pd.DataFrame([(q, *out_status(q)) for q in ALLQ], columns=["query", "output_status", "output_rows"]).set_index("query")
EMPTY = OS.index[(OS["output_status"] == "complete") & (OS["output_rows"] == 0)]
CAUSES = [
    ("wall-clock date window (NOW / CURRENT_DATE)", lambda q: bool(WALL.search(nc[q]))),
    ("fixed-date window of 31 days or less", lambda q: "2024-10-01 12:34:56" in nc[q] and min_window_days(nc[q]) <= 31),
    ("HAVING threshold", lambda q: bool(F.at[q, "has_having"])),
    ("EXISTS / IN (SELECT)", lambda q: bool(F.at[q, "has_exists"] or F.at[q, "has_in_subq"])),
    ("LIKE pattern", lambda q: bool(F.at[q, "has_like"])),
    ("lookup table (tags, links, history/close types)",
     lambda q: bool(F.loc[q, ["tbl_tags", "tbl_linktypes", "tbl_postlinks", "tbl_posthistorytypes",
                              "tbl_closereasontypes"]].any())),
    ("5 or more joins", lambda q: F.at[q, "n_join"] >= 5),
]


def likely(q):
    return next((lbl for lbl, f in CAUSES if f(q)), "none of the listed features")


EM = pd.DataFrame(index=pd.Index(EMPTY, name="query"))
EM["in_query_energy"] = EM.index.isin(OK)
EM["likely_reason"] = [likely(q) for q in EM.index]
EM["wallclock_dependent"] = [bool(WALL.search(nc[q])) for q in EM.index]
EM["date_window_days"] = [min_window_days(nc[q]) for q in EM.index]
EM["duplicate_queries"] = DS.loc[EM.index, "duplicate_queries"]
EM["energy_j"] = Q["energy_j"].reindex(EM.index).round(4)
out_con = OUT_CONSOLE.read_text("utf-8", "replace") if OUT_CONSOLE.exists() else ""
OUT_START = (re.search(r"detach\.sh start (\S+)", out_con) or [None, "unknown"])[1]
print(f"output run ({OUT_START}): {int((OS['output_status'] == 'complete').sum())} complete, "
      f"{int((OS['output_status'] == 'capped at 1000 rows').sum())} capped, {int((OS['output_status'] == 'timeout').sum())} "
      f"timeouts;  empty: {len(EM)} ({int(EM['in_query_energy'].sum())} of them in query_energy.csv)")
print(EM["likely_reason"].value_counts().to_string())

output run (2026-09-29T10:07:21Z): 9712 complete, 2074 capped, 8 timeouts;  empty: 944 (911 of them in query_energy.csv)
likely_reason
none of the listed features                        210
wall-clock date window (NOW / CURRENT_DATE)        189
5 or more joins                                    142
fixed-date window of 31 days or less               126
lookup table (tags, links, history/close types)    121
EXISTS / IN (SELECT)                                64
HAVING threshold                                    61
LIKE pattern                                        31


## 4. Run, machine and dataset information

From the run's `summary.txt` and `query_runinfo` (written by the harness), the database catalogue
logged at the start of the run, and a few read-only system commands on laptop2 (OS, disk, memory,
PostgreSQL config file, query-file checksums).

In [6]:
def kv(text):
    out = {}
    for line in text.splitlines():
        m = re.match(r"^\s*([A-Za-z_][\w ]*?):\s+(.*\S)\s*$", line)
        if m:
            out[m.group(1).strip()] = m.group(2)
    return out


S = kv(SUMMARY)
RI = pd.read_csv(RUN / f"query_runinfo_{DB}.csv")
ri = RI.iloc[0]
ri_const = {c: RI[c].nunique() == 1 for c in RI.columns if c not in ("timestamp_utc", "run_id")}
sysinfo = (CACHE / f"{RUN_ID}.sysinfo.txt").read_text("utf-8", "replace").split("##")
host_, osr, kern, mem, disk, pgconf, confd, md5s = [s.strip() for s in sysinfo[:8]]
osname = dict(re.findall(r'^(\w+)="?([^"\n]*)"?', osr, re.M)).get("PRETTY_NAME", "")
mem_b = int(re.search(r"Mem:\s+(\d+)", mem).group(1))
disk_l = disk.splitlines()[1] if len(disk.splitlines()) > 1 else disk
pgset = dict(re.findall(r"^(\w+)\s*=\s*'?([^'#\n]*?)'?\s*(?:#.*)?$", pgconf, re.M))
rem_md5 = {l.split()[1]: l.split()[0] for l in md5s.splitlines() if l.endswith(".sql")}
loc_md5 = {f"{q}.sql": hashlib.md5((QDIR / f"{q}.sql").read_bytes()).hexdigest() for q in ALLQ}
QFILES_MATCH = rem_md5 == loc_md5
CAT = pd.read_csv(RUN / f"query_catalog_{DB}.csv").drop_duplicates("relname", keep="first")
CAT1 = pd.read_csv("cache/sqlstorm_cache/stackoverflow/logs/query_catalog_stackoverflow_1gb.csv").drop_duplicates("relname", keep="last")
same_db = CAT.set_index("relname")["total_bytes"].equals(CAT1.set_index("relname")["total_bytes"].reindex(CAT["relname"]))
header = SQL_RAW[ALLQ[0]].splitlines()[0]
src = re.search(r"SQLStorm (v[\d.]+)/(\S+?)/queries/\S+ @ (\w+)", header)
start = pd.Timestamp(S["started_utc"]); finish = pd.Timestamp(S["finished_utc"])

INFO = [
    ("dataset", "name", "SQLStorm StackOverflow queries on the StackOverflow (dba.stackexchange) 1 GB database", "README"),
    ("dataset", "query_source", f"SQLStorm {src.group(1)}, {src.group(2)}/queries, commit {src.group(3)}", "query file header"),
    ("dataset", "query_id", "the file name number: query 15407 = 15407.sql", "README"),
    ("dataset", "queries_in_run", len(ALLQ), "run order"),
    ("dataset", "queries_fully_successful", len(OK), "this notebook"),
    ("dataset", "distinct_sql_texts", len(U), "this notebook"),
    ("dataset", "query_files_identical_to_laptop2", QFILES_MATCH, "md5 of all .sql files"),
    ("dataset", "executed_statement", "EXPLAIN (ANALYZE, TIMING OFF, COSTS ON, SUMMARY ON, BUFFERS) <query>", "query files"),
    ("database", "name", DB, "summary.txt"),
    ("database", "tables", int((CAT["relkind"] == "r").sum()), "query_catalog"),
    ("database", "total_size_mb", round(CAT.loc[CAT["relkind"] == "r", "total_bytes"].sum() / 2 ** 20, 1), "query_catalog"),
    ("database", "same_size_as_output_run_database", same_db, "catalog of both machines"),
    ("run", "run_id", S["run_id"], "summary.txt"),
    ("run", "started_utc", S["started_utc"], "summary.txt"),
    ("run", "finished_utc", S["finished_utc"], "summary.txt"),
    ("run", "duration_h", round((finish - start).total_seconds() / 3600, 2), "summary.txt"),
    ("run", "harness", "GreenSQL run_warm_stepup.sh / query_runner (make warm-stepup)", "console.log"),
    ("run", "batch_sizes", S["batch_sizes"], "summary.txt"),
    ("run", "measured_runs_per_size", S["runs_per_size"], "summary.txt"),
    ("run", "warmups", S["warmup"], "summary.txt"),
    ("run", "warmup_fail_skip", S["warmup_fail_skip"], "summary.txt"),
    ("run", "repeats", S["repeats"], "summary.txt"),
    ("run", "statement_timeout", S["statement_timeout"], "summary.txt"),
    ("run", "batch_cap_slow", S["batch_cap_slow"], "summary.txt"),
    ("run", "query_order", "random (one fixed shuffle, saved run order)", "run_order"),
    ("run", "thermal_policy", S["thermal_policy"], "summary.txt"),
    ("run", "clock", f"fixed, max {int(S['clock_max_khz']) // 1000} MHz, governor {S['clock_governor']}", "summary.txt"),
    ("run", "rapl_power_limits", f"PL1 {S['rapl_pl1_w']} W (tau {S['rapl_pl1_tau_s']} s), PL2 {S['rapl_pl2_w']} W", "summary.txt"),
    ("run", "throttle_events", S["dmesg_throttle_msgs"], "summary.txt"),
    ("run", "energy_counter", ri["rapl"], "query_runinfo"),
    ("run", "parallel_workers", S["workers"], "summary.txt"),
    ("machine", "product", S["product"], "summary.txt"),
    ("machine", "cpu", S["cpu_model"], "summary.txt"),
    ("machine", "cpu_threads", S["cpu_count"], "summary.txt"),
    ("machine", "microcode", S["microcode"], "summary.txt"),
    ("machine", "bios", S["bios"], "summary.txt"),
    ("machine", "memory_gb", round(mem_b / 2 ** 30, 1), "free -b"),
    ("machine", "disk", re.sub(r"\s+", " ", disk_l), "lsblk"),
    ("machine", "os", osname, "/etc/os-release"),
    ("machine", "kernel", S["kernel"], "summary.txt"),
    ("machine", "cpufreq_driver", S["freq_driver"], "summary.txt"),
    ("machine", "machine_identity_constant_during_run", all(ri_const.values()), "query_runinfo (every query)"),
    ("postgresql", "version", ri["pg_version"], "query_runinfo"),
    ("postgresql", "config", "stock postgresql.conf; conf.d " + ("empty" if not confd else confd), "postgresql.conf"),
    ("postgresql", "shared_buffers", pgset.get("shared_buffers", "default"), "postgresql.conf"),
    ("postgresql", "max_connections", pgset.get("max_connections", "default"), "postgresql.conf"),
    ("postgresql", "other_settings", "PostgreSQL 18 defaults (postgresql.auto.conf not readable without sudo)", "postgresql.conf"),
    ("labels", "energy_definition", "RAPL package energy per execution = (E16 - E1) / 15", "this notebook"),
    ("labels", "energy_se_model", f"sqrt(({1000 * NOISE_A:.1f} mJ)^2 + ({100 * NOISE_K:.2f}% * energy)^2), from {NOISE_N} "
                                  f"duplicate measurements", "this notebook"),
    ("outputs", "source", f"output_runner on a second identical laptop, same database and query files, 60 s timeout, started {OUT_START}",
     "output run console"),
]
RUNINFO = pd.DataFrame(INFO, columns=["section", "key", "value", "source"])
print(RUNINFO.to_string(index=False, max_colwidth=90))

   section                                  key                                                                                      value                      source
   dataset                                 name      SQLStorm StackOverflow queries on the StackOverflow (dba.stackexchange) 1 GB database                      README
   dataset                         query_source                                      SQLStorm v1.0, stackoverflow/queries, commit b3bb0b96           query file header
   dataset                             query_id                                              the file name number: query 15407 = 15407.sql                      README
   dataset                       queries_in_run                                                                                      11794                   run order
   dataset             queries_fully_successful                                                                                      11431               this noteboo

## 5. Write the export folder

In [7]:
EXPORT.mkdir(parents=True, exist_ok=True)
QE = Q.copy()
QE["duplicate_queries"] = DS["duplicate_queries"].reindex(QE.index)
QE["empty_output"] = QE.index.isin(EMPTY)
QE["wallclock_dependent"] = [bool(WALL.search(nc[q])) for q in QE.index]
QE["sql_hash"] = SQLH.reindex(QE.index)
QE.index.name = "query"
cols = ["energy_j", "energy_j_se", "time_s", "energy_core_j", "energy_dram_j", "fixed_overhead_j", "e1_j", "e16_j",
        "t1_s", "t16_s", "power_w", "temp_start_c", "mhz_mean", "duplicate_queries", "empty_output", "wallclock_dependent",
        "sql_hash"]
QE[cols].round({"energy_j": 5, "energy_j_se": 5, "time_s": 6, "energy_core_j": 5, "energy_dram_j": 5,
                "fixed_overhead_j": 5, "e1_j": 5, "e16_j": 5, "t1_s": 6, "t16_s": 6, "power_w": 3}).to_csv(EXPORT / "query_energy.csv")

EX = b[b["n_failed"] > 0][["reason"]].copy()
EX["duplicate_queries"] = DS["duplicate_queries"].reindex(EX.index)
EX["output_status"] = OS["output_status"].reindex(EX.index)
EX.index.name = "query"
EX.to_csv(EX_PATH := EXPORT / "excluded_queries.csv")

SMP = pd.read_csv(RUN / f"query_samples_{DB}.csv")
SMP["query"] = SMP["query"].str.extract(r"(\d+)\.sql$")[0].astype(int)
s1 = SMP[(SMP["phase"] == "measured") & (SMP["batchnum"] == 1) & SMP["query"].isin(OK)].drop_duplicates("query").set_index("query")
s16 = SMP[(SMP["phase"] == "measured") & (SMP["batchnum"] == 16) & SMP["query"].isin(OK)]
ST = s1[["server_planning_ms", "server_execution_ms", "plan_nodes", "scan_nodes", "rows_out", "rows_processed",
         "rows_estimated", "bytes_processed", "rows_removed_filter", "workers_launched", "relations", "shared_hit_blks",
         "shared_read_blks", "temp_read_blks", "temp_written_blks"]].copy()
ST["server_execution_ms_median16"] = s16.groupby("query")["server_execution_ms"].median()
ST["copies_logged_16"] = s16.groupby("query").size()
for c in ["plan_nodes", "scan_nodes", "rows_out", "rows_processed", "rows_estimated", "bytes_processed",
          "rows_removed_filter", "workers_launched", "shared_hit_blks", "shared_read_blks", "temp_read_blks",
          "temp_written_blks", "copies_logged_16"]:
    ST[c] = pd.to_numeric(ST[c], errors="coerce").round().astype("Int64")
ST["server_execution_ms_median16"] = ST["server_execution_ms_median16"].round(3)
ST.index.name = "query"
ST.sort_index().to_csv(EXPORT / "query_execution_stats.csv")

DUPSIM.to_csv(EXPORT / "duplicates_and_similar.csv")

# one row per group of byte-identical outputs; inside a group, '=' joins identical SQL, '|' separates different SQL
def sql_ratio(q1, q2):
    return difflib.SequenceMatcher(None, toks[q1], toks[q2], autojunk=False).ratio()


grp_rows = []
for h, g in og_id.groupby(og_id):
    qs = sorted(g.index)
    variants = sorted(pd.Series(qs, index=qs).groupby(CANON[qs]).apply(sorted), key=lambda v: v[0])
    cs = sorted({CANON[q] for q in qs})
    sims = [sql_ratio(a, c) for a, c in itertools.combinations(cs, 2)]
    lines_ = (ODIR / f"{qs[0]}.txt").read_text("utf-8", "replace").split("\n")
    grp_rows.append(dict(
        n_queries=len(qs), n_sql=len(variants),
        kind="same SQL (copies)" if len(variants) == 1 else
             ("different SQL" if len(variants) == len(qs) else "copies + different SQL"),
        n_in_query_energy=int(sum(q in set(OK) for q in qs)),
        rows=OS.at[qs[0], "output_rows"], capped=OS.at[qs[0], "output_status"] == "capped at 1000 rows",
        ncol=lines_[0].count("|") + 1, sim_min=round(min(sims), 4) if sims else np.nan,
        queries=" | ".join(" = ".join(map(str, v)) for v in variants), header=lines_[0], first=qs[0]))
GROUPS = pd.DataFrame(grp_rows).sort_values(["n_queries", "first"], ascending=[False, True]).drop(columns="first")
GROUPS.insert(0, "group", [f"G{i + 1}" for i in range(len(GROUPS))])
GROUPS["rows"] = GROUPS["rows"].astype("Int64")
GROUPS.to_csv(EXPORT / "duplicate_and_similar_groups.csv", index=False)
assert GROUPS["n_queries"].sum() == len(og_id)
print(f"duplicate_and_similar_groups: {len(GROUPS)} groups, {GROUPS['n_queries'].sum()} queries; "
      + ", ".join(f"{k} {v}" for k, v in GROUPS["kind"].value_counts().items()))
EM.to_csv(EXPORT / "empty_output_queries.csv")
RUNINFO.to_csv(EXPORT / "run_info.csv", index=False)
CAT.loc[:, ["relname", "relkind", "reltuples", "relpages", "heap_bytes", "index_bytes", "total_bytes"]].rename(
    columns={"relname": "relation", "relkind": "kind", "reltuples": "rows_estimate"}).replace(
    {"kind": {"r": "table", "i": "index"}}).to_csv(EXPORT / "database_tables.csv", index=False)
print(f"query_execution_stats: {len(ST)} queries, 16-copy samples complete for {int((ST['copies_logged_16'] == 16).sum())}")

duplicate_and_similar_groups: 452 groups, 3050 queries; same SQL (copies) 258, copies + different SQL 125, different SQL 69
query_execution_stats: 11386 queries, 16-copy samples complete for 11385


In [8]:
R = dict(RUNINFO.set_index("key")["value"])
lines = f"""# SQLStorm StackOverflow — energy per query (laptop2 run)

Energy and time of **{len(OK)} SQL queries** from SQLStorm's StackOverflow set, measured on one laptop
(Dell Latitude 7490, Intel i5-8350U, clock fixed at 2.5 GHz) with PostgreSQL {R['version']} on the
StackOverflow (dba.stackexchange) 1 GB database ("laptop2" is the measuring machine's name in
`run_info.csv`). The per-query CSVs are keyed by `query`.

**Query id.** `query` is the number in the SQLStorm file name: query 15407 is
`{src.group(2)}/queries/15407.sql` in SQLStorm {src.group(1)} (commit `{src.group(3)}`).
Each query was executed as `EXPLAIN (ANALYZE, TIMING OFF, COSTS ON, SUMMARY ON, BUFFERS) <query>`:
it runs to completion on the server, but its rows are not sent to the client.

## Files

| file | rows | content |
|---|---|---|
| `query_energy.csv` | {len(QE)} | every fully successful query and its energy per execution — **the labels** |
| `excluded_queries.csv` | {len(EX)} | queries of the run that are not in `query_energy.csv`, and why |
| `query_execution_stats.csv` | {len(ST)} | PostgreSQL execution statistics of each successful query |
| `duplicates_and_similar.csv` | {len(DUPSIM)} | duplicates, near-duplicates and same-output queries |
| `duplicate_and_similar_groups.csv` | {len(GROUPS)} | groups of queries with identical output, and the SQL variants in each |
| `empty_output_queries.csv` | {len(EM)} | queries whose result is empty |
| `run_info.csv` | {len(RUNINFO)} | run parameters, machine, OS, PostgreSQL, dataset provenance |
| `database_tables.csv` | {len(CAT)} | tables and indexes of the database, row counts and sizes |

## How the energy was measured

Every query in the set ({len(ALLQ)}) was run once, in a fixed random order, over {R['duration_h']} h
({R['started_utc']} to {R['finished_utc']}). For each query:

1. thermal gate: wait until the CPU package is between 40 and 60 °C;
2. two single-copy warm-ups (the first fills the cache);
3. a measured batch of **1** copy and a measured batch of **16** back-to-back copies, each recording
   wall time and RAPL energy (package, cores, DRAM).

The energy of one execution is the **slope** `energy_j = (E16 − E1) / 15`. It removes what each
batch pays once (starting the client, connecting, batch edges), leaving what one more execution of the
query costs. Time per execution `time_s` is the same slope on wall time. No idle power was subtracted:
`energy_j` is the whole package energy while the query runs. Statement timeout {R['statement_timeout']}
per copy; parallel query at the planner default.

**Fully successful** means none of the four batches failed. {len(EX)} queries are excluded:
{'; '.join(f'{v} {k}' for k, v in EX['reason'].str.replace(r'warm-up \d', 'warm-up', regex=True).value_counts().items())}.

**Precision.** {NOISE_N} queries are exact copies of other queries ({NOISE_SETS} sets) and were measured
independently, which gives the repeat noise of one slope on this machine:
`energy_j_se = sqrt(({1000 * NOISE_A:.1f} mJ)^2 + ({100 * NOISE_K:.2f}% x energy_j)^2)`, about
{100 * NOISE_K:.1f}% for most queries. It was fitted on slopes up to {NOISE_MAX:.0f} J and is
extrapolated above. The noise has a heavy tail: {100 * NOISE_TAIL:.1f}% of the copies lie more than 3
standard errors from their set's mean (the worst set spreads by {100 * NOISE_REL_MAX:.0f}%).

## `query_energy.csv`

| column | unit | meaning |
|---|---|---|
| `query` | — | SQLStorm query number (file name) |
| `energy_j` | J | **package energy per execution**, `(e16_j − e1_j) / 15` |
| `energy_j_se` | J | estimated standard error of `energy_j` (repeat noise model above) |
| `time_s` | s | wall time per execution, `(t16_s − t1_s) / 15` |
| `energy_core_j` | J | CPU-core part of `energy_j` (RAPL core domain), same slope |
| `energy_dram_j` | J | DRAM energy per execution (separate RAPL domain, not included in `energy_j`) |
| `fixed_overhead_j` | J | per-batch fixed cost, `e1_j − energy_j` |
| `e1_j`, `e16_j` | J | measured package energy of the 1-copy and the 16-copy batch |
| `t1_s`, `t16_s` | s | measured wall time of the two batches |
| `power_w` | W | mean package power during the 16-copy batch |
| `temp_start_c` | °C | package temperature at the start of the 16-copy batch |
| `mhz_mean` | MHz | mean clock over all 8 logical CPUs during the 16-copy batch (idle CPUs included, so below the 2.5 GHz ceiling) |
| `duplicate_queries` | — | the other queries whose SQL is identical to this one (`;`-separated; empty if none) |
| `empty_output` | — | the query returns no rows |
| `wallclock_dependent` | — | filters on `NOW()` / `CURRENT_DATE`, so its result and cost depend on the run date |
| `sql_hash` | — | md5 of the normalised SQL text (see below), to check a query file is the same one |

## `query_execution_stats.csv`

From PostgreSQL's `EXPLAIN ANALYZE` output of the measured single copy: planning and execution time
(ms), number of plan and scan nodes, rows returned (`rows_out`), rows processed by all nodes,
the planner's row estimate, bytes processed, rows removed by filters, parallel workers launched,
relations touched (as PostgreSQL names them, so CTE and alias names appear too), shared-buffer hits and reads, temp blocks read and written (8 kB blocks);
`server_execution_ms_median16` is the median over the 16-copy batch. The harness logged no per-copy
samples for {len(QE) - len(ST)} of the successful queries, so they are absent from this file. These are measured during
execution — useful as targets or for analysis, but not available before a query runs.

## `duplicates_and_similar.csv`

How queries relate to each other in form (SQL text) and in output, for every query that has at least
one such relation. Nothing was removed or merged because of it: all queries keep their own
measurement in `query_energy.csv`. SQL is normalised before comparing: the SQLStorm header comment and
`EXPLAIN (…)` wrapper are removed, comments removed, whitespace collapsed, lower-cased.

| column | meaning |
|---|---|
| `in_query_energy` | the query is in `query_energy.csv` |
| `sql_hash` | md5 of the normalised SQL |
| **form: identical** | |
| `duplicate_queries` | the other queries whose SQL is identical to this one (`;`-separated) |
| `duplicate_count` | how many there are |
| **form: similar** | |
| `similar_queries` | all queries whose SQL differs from this one but has token 3-gram Jaccard similarity ≥ {J_MIN} (`;`-separated) |
| `similar_count` | how many there are |
| `most_similar_query`, `most_similar_jaccard` | the closest of them and its similarity |

All three use the same form: a list of the other queries in the relation and its length. The
relations are symmetric — if 15027 lists 15043, then 15043 lists 15027.
| **output** | |
| `same_output_queries` | the other queries whose result file is byte-identical to this one's (header, every row, row count; `;`-separated) |
| `same_output_count` | how many there are |

{len(ALLQ)} queries contain {len(U)} distinct SQL texts; {int(DS['similar_count'].notna().sum())} queries have
at least one similar query, and {len(og_id)} share their exact output with another query. Similar
queries often differ in a single clause (a LIMIT, a filter, a column), and their energies can differ
widely; copies of identical SQL were measured independently.

## `duplicate_and_similar_groups.csv`

The same information grouped: one row per group of queries whose result files are byte-identical
({len(GROUPS)} groups, {GROUPS['n_queries'].sum()} queries). Every set of identical-SQL queries falls
inside one of these groups, since copies of a query always returned the same output.

| column | meaning |
|---|---|
| `group` | group id, G1 = largest |
| `n_queries` | queries in the group |
| `n_sql` | distinct SQL texts among them (normalised as above) |
| `kind` | `same SQL (copies)`, `different SQL`, or `copies + different SQL` |
| `n_in_query_energy` | how many of the group's queries are in `query_energy.csv` |
| `rows` | rows in the shared result (empty when capped) |
| `capped` | the result had 1000 rows or more (the output run kept the first 1000) |
| `ncol` | columns in the result |
| `sim_min` | lowest token similarity between two of the group's distinct SQL texts (difflib ratio, 1 = same text) |
| `queries` | the group's queries: `=` joins identical SQL, `|` separates different SQL texts, e.g. `15027 = 15043 | 16207` |
| `header` | the result's column names |

## `empty_output_queries.csv`

The {len(EM)} queries whose result is empty, from a separate output run (`output_runner`, 60 s
timeout, on a second, identical Latitude 7490 with the same database — every table the same size — and
identical query files).
`likely_reason` is the first matching SQL feature (a heuristic). `wallclock_dependent` queries filter
on `NOW()`/`CURRENT_DATE`; the data ends in 2024, so their date window is empty when run now, they
skip most of their work, and their energy is lower than the SQL suggests. `date_window_days` is the
shortest `INTERVAL` in the query.

## Caveats

* One machine, one run: each query was measured once (one 1-copy and one 16-copy batch).
* Energy is measured while running `EXPLAIN ANALYZE`, without sending rows to the client.
* `wallclock_dependent` queries ({int(QE['wallclock_dependent'].sum())} in `query_energy.csv`) would cost
  differently on another date.
* Excluded queries are not missing at random: they are the slowest (they hit the 15 s timeout).
"""
(EXPORT / "README.md").write_text(lines, encoding="utf-8")
for f in sorted(EXPORT.iterdir()):
    print(f"{f.name:32} {f.stat().st_size / 1024:8.1f} KB")

database_tables.csv                   1.3 KB
duplicate_and_similar_groups.csv     67.8 KB
duplicates_and_similar.csv         4361.6 KB
empty_output_queries.csv             57.7 KB
excluded_queries.csv                 26.8 KB
query_energy.csv                   2023.3 KB
query_execution_stats.csv          1236.7 KB
README.md                             9.0 KB
run_info.csv                          3.0 KB


## 6. Checks on the export

In [9]:
chk = pd.read_csv(EXPORT / "query_energy.csv")
assert chk["query"].is_unique and len(chk) == len(OK)
assert np.allclose(chk["energy_j"], (chk["e16_j"] - chk["e1_j"]) / 15, atol=1e-4)
assert set(chk["query"]).isdisjoint(pd.read_csv(EXPORT / "excluded_queries.csv")["query"])
assert len(chk) + len(EX) == len(ALLQ)
assert (chk["energy_j"] > 0).all() and (chk["time_s"] > 0).all()
print("query_energy.csv: one row per successful query, slope recomputes, no overlap with excluded, all positive")
print(chk[["energy_j", "energy_j_se", "time_s", "power_w"]].describe().round(3).to_string())

query_energy.csv: one row per successful query, slope recomputes, no overlap with excluded, all positive
        energy_j  energy_j_se     time_s    power_w
count  11431.000    11431.000  11431.000  11431.000
mean       7.495        0.067      1.108      8.241
std       12.356        0.110      1.624      3.125
min        0.012        0.001      0.002      4.788
25%        1.481        0.013      0.153      5.201
50%        3.974        0.035      0.615      7.031
75%        8.632        0.077      1.311     11.215
max      225.891        2.006     14.452     16.302
